![Two data scientists working on a dashboard.](hr-image-small.png)

A common problem when creating models to generate business value from data is that the datasets can be so large that it can take days for the model to generate predictions. Ensuring that your dataset is stored as efficiently as possible is crucial for allowing these models to run on a more reasonable timescale without having to reduce the size of the dataset.

You've been hired by a major online data science training provider called *Training Data Ltd.* to clean up one of their largest customer datasets. This dataset will eventually be used to predict whether their students are looking for a new job or not, information that they will then use to direct them to prospective recruiters.

You've been given access to `customer_train.csv`, which is a subset of their entire customer dataset, so you can create a proof-of-concept of a much more efficient storage solution. The dataset contains anonymized student information, and whether they were looking for a new job or not during training:

| Column                   | Description                                                                      |
|------------------------- |--------------------------------------------------------------------------------- |
| `student_id`             | A unique ID for each student.                                                    |
| `city`                   | A code for the city the student lives in.                                        |
| `city_development_index` | A scaled development index for the city.                                         |
| `gender`                 | The student's gender.                                                            |
| `relevant_experience`    | An indicator of the student's work relevant experience.                          |
| `enrolled_university`    | The type of university course enrolled in (if any).                              |
| `education_level`        | The student's education level.                                                   |
| `major_discipline`       | The educational discipline of the student.                                       |
| `experience`             | The student's total work experience (in years).                                  |
| `company_size`           | The number of employees at the student's current employer.                       |
| `company_type`           | The type of company employing the student.                                       |
| `last_new_job`           | The number of years between the student's current and previous jobs.             |
| `training_hours`         | The number of hours of training completed.                                       |
| `job_change`             | An indicator of whether the student is looking for a new job (`1`) or not (`0`). |

In [20]:
# Import necessary libraries
import pandas as pd
import numpy as np
# Load the dataset
ds_jobs = pd.read_csv("customer_train.csv")

# View the dataset
ds_jobs.head(25)

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0
5,21651,city_176,0.764,NaN,Has relevant experience,Part time course,Graduate,STEM,11,NaN,NaN,1,24,1.0
6,28806,city_160,0.920,Male,Has relevant experience,no_enrollment,High School,NaN,5,50-99,Funded Startup,1,24,0.0
7,402,city_46,0.762,Male,Has relevant experience,no_enrollment,Graduate,STEM,13,<10,Pvt Ltd,>4,18,1.0
8,27107,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,7,50-99,Pvt Ltd,1,46,1.0
9,699,city_103,0.920,NaN,Has relevant experience,no_enrollment,Graduate,STEM,17,10000+,Pvt Ltd,>4,123,0.0


In [23]:
# Create a copy of ds_jobs for transforming
ds_jobs_transformed = ds_jobs.copy()

# Start coding here. Use as many cells as you like!
# print(ds_jobs_transformed.info()) shows this
"""
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19158 entries, 0 to 19157
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   student_id              19158 non-null  int64  
 1   city                    19158 non-null  object 
 2   city_development_index  19158 non-null  float64
 3   gender                  14650 non-null  object 
 4   relevant_experience     19158 non-null  object 
 5   enrolled_university     18772 non-null  object 
 6   education_level         18698 non-null  object 
 7   major_discipline        16345 non-null  object 
 8   experience              19093 non-null  object 
 9   company_size            13220 non-null  object 
 10  company_type            13018 non-null  object 
 11  last_new_job            18735 non-null  object 
 12  training_hours          19158 non-null  int64  
 13  job_change              19158 non-null  float64
dtypes: float64(2), int64(2), object(10)
memory usage: 2.0+ MB
"""
"""
Instructions have set the following requirements:

* Columns containing categories with only two factors must be stored as Booleans (bool).
* Columns containing integers only must be stored as 32-bit integers (int32).
* Columns containing floats must be stored as 16-bit floats (float16).
* Columns containing nominal categorical data must be stored as the category data type.
* Columns containing ordinal categorical data must be stored as ordered categories, and not mapped to numericalvalues, with an order that reflects the natural order of the column.
* The DataFrame should be filtered to only contain students with 10 or more years of experience at companies with at least 1000 employees, as their recruiter base is suited to more experienced professionals at enterprise companies.
"""
"""----------------------------------------------------------------------------------------------------------------"""
#Company Size: Transform into categorical
#print(ds_jobs_transformed["company_size"].value_counts(dropna=False)) 
"""
NaN          5938
<10          1308
10-49        1471
50-99        3083
100-499      2571
500-999       877
   ---Cut off---
1000-4999    1328 
5000-9999     563
10000+       2019
"""
#Order: "Unknown","<10","10-49","50-99","100-499","500-999","1000-4999","5000-9999","10000+"
#transform NaN to "Unknown" category
ds_jobs_transformed["company_size"]=ds_jobs_transformed["company_size"].fillna("Unknown")
ds_jobs_transformed["company_size"]=ds_jobs_transformed["company_size"].astype("category")
ds_jobs_transformed["company_size"]=ds_jobs_transformed["company_size"].cat.reorder_categories(\
    new_categories=["Unknown","<10","10-49","50-99","100-499","500-999","1000-4999","5000-9999","10000+"],ordered=True)
#print(ds_jobs_transformed["company_size"].value_counts(dropna=False).sort_index())
"""----------------------------------------------------------------------------------------------------------------"""
#education_level into category and order it
#print(ds_jobs_transformed["education_level"].value_counts(dropna=False)) reveal the following
"""
NaN                 460
Primary School      308
High School        2017
Graduate          11598
Masters            4361
Phd                 414
"""
#Order: "Unknown","Primary School","High School","Graduate","Masters","Phd"
#transform NaN to "Unknown" and make into a category
ds_jobs_transformed["education_level"]=ds_jobs_transformed["education_level"].fillna("Unknown")
ds_jobs_transformed["education_level"]=ds_jobs_transformed["education_level"].astype("category")
ds_jobs_transformed["education_level"]=ds_jobs_transformed["education_level"].cat.reorder_categories(\
    new_categories=["Unknown","Primary School","High School","Graduate","Masters","Phd"],ordered=True)

#print(ds_jobs_transformed["education_level"].cat.ordered)
"""
Unknown             460
Primary School      308
High School        2017
Graduate          11598
Masters            4361
Phd                 414
"""
#Data is ordered and NaN is turned into "Unknown" category 
"""----------------------------------------------------------------------------------------------------------------"""
#Gender: keep all values and turn into category.  This is what value_counts(dropna=False) reveal
"""
Male       13221
NaN         4508
Female      1238
Other        191
"""
ds_jobs_transformed["gender"]=ds_jobs_transformed["gender"].fillna("Unknown")
ds_jobs_transformed["gender"]=ds_jobs_transformed["gender"].astype("category")

#print(ds_jobs_transformed["gender"].value_counts(dropna=False))
"""----------------------------------------------------------------------------------------------------------------"""
#Relevant Experience: turn into bool column.  Here's value_counts(dropna=False)
#print(ds_jobs_transformed["relevant_experience"].value_counts(dropna=False))
"""
Has relevant experience   13792
No relevant experience     5366
"""
ds_jobs_transformed["relevant_experience"]= ds_jobs_transformed["relevant_experience"].replace({\
    "Has relevant experience":True, "No relevant experience":False})
#print(ds_jobs_transformed["relevant_experience"].value_counts(dropna=False))
"""----------------------------------------------------------------------------------------------------------------"""
# enrolled university: transform into category and order it from unknown < no_enrollment < Part time course < Full time
# also, rename no_enrollment -> No enrollment
"""
no_enrollment       13817
Full time course     3757
Part time course     1198
NaN                   386
"""
#print(ds_jobs_transformed["enrolled_university"].value_counts(dropna=False))
ds_jobs_transformed["enrolled_university"]=ds_jobs_transformed["enrolled_university"].fillna("Unknown")
ds_jobs_transformed["enrolled_university"]=ds_jobs_transformed["enrolled_university"].astype("category")
ds_jobs_transformed["enrolled_university"]=ds_jobs_transformed["enrolled_university"].cat.rename_categories({\
    "no_enrollment":"No Enrollment"})
ds_jobs_transformed["enrolled_university"]=ds_jobs_transformed["enrolled_university"].cat.reorder_categories([\
"Unknown","No Enrollment","Part time course","Full time course"],ordered=True)
#print(ds_jobs_transformed["enrolled_university"].value_counts(dropna=False).sort_index())
"""----------------------------------------------------------------------------------------------------------------"""
#job_change: turn into bool column; similar to relevant experience.  here's what value_counts(dropna=False) reveals
#print(ds_jobs_transformed["job_change"].value_counts(dropna=False))
"""
0        14381
1         4777
"""
ds_jobs_transformed["job_change"]= ds_jobs_transformed["job_change"].replace({\
    1:True, 0:False})
#print(ds_jobs_transformed["job_change"].value_counts(dropna=False))
"""----------------------------------------------------------------------------------------------------------------"""
# last_new_job: Make it a category and make categories in ascending order.  Turn Nan to Unknown.
# print(ds_jobs_transformed["last_new_job"].value_counts(dropna=False)) shows this
"""
1        8040
>4       3290
2        2900
never    2452
4        1029
3        1024
NaN       423
"""
# Also, change "never" category -> "Never"
# Order: "Unknown","Never","1","2","3","4",">4"
ds_jobs_transformed["last_new_job"]=ds_jobs_transformed["last_new_job"].fillna("Unknown")
ds_jobs_transformed["last_new_job"]=ds_jobs_transformed["last_new_job"].astype("category")
ds_jobs_transformed["last_new_job"]=ds_jobs_transformed["last_new_job"].cat.rename_categories({\
    "never":"Never"})
ds_jobs_transformed["last_new_job"]=ds_jobs_transformed["last_new_job"].cat.reorder_categories(\
    new_categories=["Unknown","Never","1","2","3","4",">4"],ordered=True)
#print(ds_jobs_transformed["last_new_job"].value_counts(dropna=False).sort_index())

"""----------------------------------------------------------------------------------------------------------------"""
#company_type: Make it a category and make null into "Unknown" category.  Here's what value_counts(dropna=False) shows
"""
Pvt Ltd                9817
NaN                    6140
Funded Startup         1001
Public Sector           955
Early Stage Startup     603
NGO                     521
Other                   121
"""
ds_jobs_transformed["company_type"]=ds_jobs_transformed["company_type"].fillna("Unknown")
ds_jobs_transformed["company_type"]=ds_jobs_transformed["company_type"].astype("category")
#print(ds_jobs_transformed["company_type"].value_counts(dropna=False))


#major_discipline: turn into category and change "Business Degree" to "Business"
#print(ds_jobs_transformed["major_discipline"].value_counts(dropna=False))
"""
STEM               14492
NaN                 2813
Humanities           669
Other                381
Business Degree      327
Arts                 253
No Major             223
"""

ds_jobs_transformed["major_discipline"]=ds_jobs_transformed["major_discipline"].fillna("Unknown")
ds_jobs_transformed["major_discipline"]=ds_jobs_transformed["major_discipline"].astype("category")
ds_jobs_transformed["major_discipline"]=ds_jobs_transformed["major_discipline"].cat.rename_categories({"Business Degree":"Business"})
#print(ds_jobs_transformed["major_discipline"].value_counts(dropna=False))

#experience: turn into categories and order it in ascending order.  Here's what value_counts(dropna=False) show
"""
>20    3286
5      1430
4      1403
3      1354
6      1216
2      1127
7      1028
10      985
9       980
8       802
15      686
11      664
14      586
1       549
<1      522
16      508
12      494
13      399
17      342
19      304
18      280
20      148
NaN      65
"""
#print(ds_jobs_transformed["experience"].value_counts(dropna=False))
ds_jobs_transformed["experience"].fillna("Unknown",inplace=True)
ds_jobs_transformed["experience"]=ds_jobs_transformed["experience"].astype("category")
ds_jobs_transformed["experience"]=ds_jobs_transformed["experience"].cat.reorder_categories(\
    new_categories=["Unknown","<1","1","2","3","4","5","6","7","8","9","10"
                    ,"11","12","13","14","15","16","17","18","19","20",">20"]
                    ,ordered=True)
#ds_jobs_transformed["experience"].value_counts(dropna=False)


#city: turn into category.  Order is unnecessary. 
ds_jobs_transformed["city"]=ds_jobs_transformed["city"].astype("category")

#student_id turn into int32
ds_jobs_transformed["student_id"]=ds_jobs_transformed["student_id"].astype("int32")

#training hours: turn into int32
ds_jobs_transformed["training_hours"]=ds_jobs_transformed["training_hours"].astype("int32")

#city_development_index: turn into float16
ds_jobs_transformed["city_development_index"]=ds_jobs_transformed["city_development_index"].astype("float16")

#print(ds_jobs_transformed.info())
"""
Comparing Final Results
Before:
    dtypes: float64(2), int64(2), object(10)
    memory usage: 2.0+ MB
After:
    dtypes: bool(2), category(9), float16(1), int32(2)
    memory usage: 400.8 KB
"""
#All Null Values handled.  Appropriate categories are ordered.  Appropriate categories are transformed.  Time to filter
#instructions want students with 10 or more years of experience at companies with at least 1000 employees
ds_jobs_transformed=ds_jobs_transformed[\
    (ds_jobs_transformed["company_size"]>="1000-4999") &\
    (ds_jobs_transformed["experience"]>="10")\
]
#print(ds_jobs_transformed.info()) shows the following
"""
Int64Index: 2201 entries, 9 to 19143
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   student_id              2201 non-null   int32   
 1   city                    2201 non-null   category
 2   city_development_index  2201 non-null   float16 
 3   gender                  2201 non-null   category
 4   relevant_experience     2201 non-null   bool    
 5   enrolled_university     2201 non-null   category
 6   education_level         2201 non-null   category
 7   major_discipline        2201 non-null   category
 8   experience              2201 non-null   category
 9   company_size            2201 non-null   category
 10  company_type            2201 non-null   category
 11  last_new_job            2201 non-null   category
 12  training_hours          2201 non-null   int32   
 13  job_change              2201 non-null   bool    
dtypes: bool(2), category(9), float16(1), int32(2)
memory usage: 70.1 KB

    All columns are either  bool, int32, float16, or category
"""
#print(ds_jobs_transformed["experience"].value_counts(dropna=False).sort_index()) shows the following
"""
Unknown      0
<1           0
1            0
2            0
3            0
4            0
5            0
6            0
7            0
8            0
9            0
10         245
11         149
12         107
13         103
14         147
15         172
16         128
17          91
18          68
19          83
20          49
>20        859
"""
#print(ds_jobs_transformed["company_size"].value_counts(dropna=False).sort_index()) shows the following
"""
Unknown         0
<10             0
10-49           0
50-99           0
100-499         0
500-999         0
1000-4999     796
5000-9999     310
10000+       1095
"""
#results show that remaining entries have 10 or more years of experience and at least 1000 employees

True
Unknown               386
No Enrollment       13817
Part time course     1198
Full time course     3757
Name: enrolled_university, dtype: int64


'\nUnknown         0\n<10             0\n10-49           0\n50-99           0\n100-499         0\n500-999         0\n1000-4999     796\n5000-9999     310\n10000+       1095\n'